In [1]:
import json
import os
from collections import defaultdict

import numpy as np
import pandas as pd

from models.SiamABC.tracker.tracker_setup import get_tracker
from models.SiamRAM import SiamRAMTracker
from utils.hydra import load_hydra_config_from_path
from vis.test_model import run_inference

with open("/home/moha/AIC-4/data_competition/metadata/contestant_manifest.json", "r") as f:
    manifest = json.load(f)
test_public_lb = manifest["public_lb"]
manifest.keys()

dict_keys(['train', 'public_lb'])

In [2]:
# """
# run_inference_gflops.py
# ~~~~~~~~~~~~~~~~~~~~~~~
# Drop-in replacement for run_inference.py.

# GFLOPs approach
# ───────────────
# We profile one full tracker.update() call on frame 1 using torch.profiler
# (built into PyTorch >= 1.8, no extra install needed).  This captures every
# torch op the whole system fires — backbone, neck, head, correlation, any YOLO
# sub-network — without knowing anything about the internal architecture.

# The profiled frame is then fed back into the normal loop so no frame is skipped.

# The final report gains a GFLOPs section showing per-frame cost, total cost,
# and effective GFLOP/s based on measured mean latency.

# Note: torch.profiler counts FLOPs for supported ops only (conv, matmul, etc.).
# Custom or fused ops contribute 0, so the number is a lower bound — the same
# convention used by most tracking papers.
# """

# import cv2
# import gc
# import os

# import cv2
# import numpy as np

# from utils.utils import _iou

# PANEL_W = 240
# GAP     = 8
# FONT    = cv2.FONT_HERSHEY_SIMPLEX

# C_GT     = (255, 120, 0)
# C_SEARCH = (0, 200, 255)
# C_UPDATE = (0, 200, 80)
# C_FRAME  = (180, 180, 180)


# # ──────────────────────────────────────────────────────────────────────────────
# # Full-system GFLOPs measurement
# # ──────────────────────────────────────────────────────────────────────────────
# # ── corrected GFLOPs helper ───────────────────────────────────────────────────
# def _measure_gflops(net, tracking_config, device="cuda"):
#     """
#     Returns (feat_gflops, track_gflops) for one forward pass of the network.
#     Returns (0.0, 0.0) on failure.
#     """
#     try:
#         import torch                          # ✅ add this
#         from thop import profile as thop_profile

#         t_size  = tracking_config["template_size"]   # e.g. 128
#         s_size  = tracking_config["instance_size"]   # e.g. 256

#         template         = torch.randn(1, 3, t_size, t_size).to(device)
#         dynamic_template = torch.randn(1, 3, t_size, t_size).to(device)
#         search           = torch.randn(1, 3, s_size, s_size).to(device)
#         dynamic_search   = torch.randn(1, 3, s_size, s_size).to(device)

#         net.eval()

#         # ── feature extractor cost (one crop) ────────────────────────────────
#         macs_feat, _ = thop_profile(
#             net.encoder,
#             inputs=(search,),          # single tensor → single positional arg ✓
#             verbose=False,
#         )

#         # ── full forward pass ─────────────────────────────────────────────────
#         # forward(self, x) takes ONE argument x=(t, dt, s, ds)
#         # so inputs must be a 1-tuple whose element IS that 4-tuple:
#         #   profile calls  net( *inputs )  →  net( (t, dt, s, ds) )  ✓
#         x = (template, dynamic_template, search, dynamic_search)
#         macs_full, _ = thop_profile(
#             net,
#             inputs=(x,),               # ← the extra comma makes it a 1-tuple
#             verbose=False,
#         )

#         gflops_feat = macs_feat  * 2 / 1e9
#         gflops_full = macs_full  * 2 / 1e9
#         return gflops_feat, gflops_full

#     except Exception as e:
#         print(f"[GFLOPs] thop profiling failed: {e}")
#         return 0.0, 0.0
#     # ──────────────────────────────────────────────────────────────────────────────
# # Visualisation helpers  (unchanged from original)
# # ──────────────────────────────────────────────────────────────────────────────



# def _measure_gflops_inference(net, tracking_config, device="cuda"):
#     """
#     Profiles ONLY what runs every frame during inference:
#       1. net.get_features(search_256)  →  encoder + neck on search crop
#       2. net.track(sf, cached_dsf, cached_tf, cached_dtf)  →  attention + head

#     Template encoder (128×128) is intentionally excluded — it only runs
#     at init and on dynamic update (every N frames, default N=10).
#     """
#     try:
#         import torch
#         from thop import profile as thop_profile

#         s_size = tracking_config["instance_size"]   # 256
#         t_size = tracking_config["template_size"]   # 128

#         net.eval()

#         dummy_search   = torch.randn(1, 3, s_size, s_size).to(device)
#         dummy_template = torch.randn(1, 3, t_size, t_size).to(device)

#         # ── Step 1: get actual feature shapes via a real forward ──────────────
#         with torch.no_grad():
#             search_feat = net.get_features(dummy_search)   # shape: (1, C, h, w)
#             tmpl_feat   = net.get_features(dummy_template) # shape: (1, C, h', w')

#         # ── Step 2: profile encoder on search (runs every frame) ─────────────
#         macs_enc, _ = thop_profile(
#             net.encoder,
#             inputs=(dummy_search,),
#             verbose=False,
#         )

#         # ── Step 3: profile neck on search (runs every frame) ────────────────
#         with torch.no_grad():
#             enc_out = net.encoder(dummy_search)
#         macs_neck, _ = thop_profile(
#             net.neck,
#             inputs=(enc_out,),
#             verbose=False,
#         )

#         macs_get_features = macs_enc + macs_neck   # total for get_features(search)

#         # ── Step 4: profile net.track() with cached feature tensors ──────────
#         # net.track() takes 4 separate tensor arguments, so wrap it
#         class _TrackWrapper(torch.nn.Module):
#             def __init__(self, net):
#                 super().__init__()
#                 self.net = net
#             def forward(self, sf, dsf, tf, dtf):
#                 return self.net.track(
#                     search_features          = sf,
#                     dynamic_search_features  = dsf,
#                     template_features        = tf,
#                     dynamic_template_features= dtf,
#                 )

#         sf  = search_feat.detach()   # current-frame search features (new every frame)
#         dsf = search_feat.detach()   # cached dynamic search features (same shape)
#         tf  = tmpl_feat.detach()     # cached static template features
#         dtf = tmpl_feat.detach()     # cached dynamic template features

#         wrapper = _TrackWrapper(net).to(device)
#         # thop calls wrapper(sf, dsf, tf, dtf) — 4 separate positional args ✓
#         macs_track, _ = thop_profile(
#             wrapper,
#             inputs=(sf, dsf, tf, dtf),
#             verbose=False,
#         )

#         # ── Totals ────────────────────────────────────────────────────────────
#         total_macs      = macs_get_features + macs_track
#         gflops_search   = macs_get_features * 2 / 1e9
#         gflops_track    = macs_track        * 2 / 1e9
#         gflops_total    = total_macs        * 2 / 1e9

#         print(f"[GFLOPs]  search encoder+neck : {gflops_search:.4f} G  (every frame)")
#         print(f"[GFLOPs]  attention + head     : {gflops_track:.4f} G  (every frame)")
#         print(f"[GFLOPs]  per-frame total      : {gflops_total:.4f} G")

#         return gflops_total, gflops_search, gflops_track

#     except Exception as e:
#         print(f"[GFLOPs] profiling failed: {e}")
#         return 0.0, 0.0, 0.0




# def _measure_gflops_inference(net, tracking_config, device="cuda"):
#     try:
#         import torch
#         from fvcore.nn import FlopCountAnalysis

#         s_size = tracking_config["instance_size"]   # 256
#         t_size = tracking_config["template_size"]   # 128
#         N      = tracking_config.get("N", 10)

#         net.eval()

#         dummy_search         = torch.randn(1, 3, s_size, s_size).to(device)
#         dummy_dynamic_search = torch.randn(1, 3, s_size, s_size).to(device)
#         dummy_template       = torch.randn(1, 3, t_size, t_size).to(device)
#         dummy_dynamic_tmpl   = torch.randn(1, 3, t_size, t_size).to(device)

#         def _flops_get_features(dummy_input, label):
#             class _W(torch.nn.Module):
#                 def __init__(self, n): super().__init__(); self.n = n
#                 def forward(self, x): return self.n.get_features(x)
#             w = _W(net).to(device)
#             flops = FlopCountAnalysis(w, (dummy_input,))
#             flops.unsupported_ops_warnings(False)
#             total = flops.total()
#             print(f"\n── {label} ({dummy_input.shape[-1]}×{dummy_input.shape[-1]}) ──")
#             # per-submodule breakdown
#             by_mod = flops.by_module()
#             for name, val in sorted(by_mod.items(), key=lambda x: -x[1]):
#                 if val > 0 and name != '':
#                     print(f"   {name:<55s} {val/1e9:.4f} G")
#             print(f"   {'TOTAL':<55s} {total/1e9:.4f} G  ({total*2/1e9:.4f} GFLOPs)")
#             return total  # fvcore returns FLOPs directly (not MACs)

#         def _flops_track(sf, dsf, tf, dtf, label):
#             class _W(torch.nn.Module):
#                 def __init__(self, n): super().__init__(); self.n = n
#                 def forward(self, sf, dsf, tf, dtf):
#                     return self.n.track(
#                         search_features           = sf,
#                         dynamic_search_features   = dsf,
#                         template_features         = tf,
#                         dynamic_template_features = dtf,
#                     )
#             w = _W(net).to(device)
#             flops = FlopCountAnalysis(w, (sf, dsf, tf, dtf))
#             flops.unsupported_ops_warnings(False)
#             total = flops.total()
#             print(f"\n── {label} ──")
#             by_mod = flops.by_module()
#             for name, val in sorted(by_mod.items(), key=lambda x: -x[1]):
#                 if val > 0 and name != '':
#                     print(f"   {name:<55s} {val/1e9:.4f} G")
#             print(f"   {'TOTAL':<55s} {total/1e9:.4f} G  ({total*2/1e9:.4f} GFLOPs)")
#             return total

#         # ── Profile each component ────────────────────────────────────────────
#         flops_search     = _flops_get_features(dummy_search,         "get_features  [search 256×256]")
#         flops_dyn_search = _flops_get_features(dummy_dynamic_search, "get_features  [dyn-search 256×256]")
#         flops_dyn_tmpl   = _flops_get_features(dummy_dynamic_tmpl,   "get_features  [dyn-template 128×128]")

#         with torch.no_grad():
#             sf  = net.get_features(dummy_search)
#             dsf = net.get_features(dummy_dynamic_search)
#             tf  = net.get_features(dummy_template)
#             dtf = net.get_features(dummy_dynamic_tmpl)

#         flops_track = _flops_track(sf.detach(), dsf.detach(),
#                                    tf.detach(), dtf.detach(), "track()  [attention + head]")

#         # ── Amortized summary ─────────────────────────────────────────────────
#         # fvcore gives FLOPs directly (already ×2 vs MACs)
#         gf_search     = flops_search     / 1e9
#         gf_dyn_search = flops_dyn_search / 1e9
#         gf_dyn_tmpl   = flops_dyn_tmpl   / 1e9
#         gf_track      = flops_track      / 1e9
#         gf_total      = gf_search + gf_track + (gf_dyn_search + gf_dyn_tmpl) / N

#         print(f"\n─── Amortized GFLOPs Summary (N={N}) ────────────────────────────")
#         print(f"  [every frame]   search get_features       : {gf_search:.4f} GFLOPs")
#         print(f"  [every frame]   attention + head          : {gf_track:.4f} GFLOPs")
#         print(f"  [every {N:2d} fr]  dyn-search get_features   : {gf_dyn_search:.4f} GFLOPs"
#               f"  → {gf_dyn_search/N:.5f} amortized")
#         print(f"  [every {N:2d} fr]  dyn-template get_features : {gf_dyn_tmpl:.4f} GFLOPs"
#               f"  → {gf_dyn_tmpl/N:.5f} amortized")
#         print(f"  ────────────────────────────────────────────────────────────────")
#         print(f"  Amortized per-frame total                 : {gf_total:.4f} GFLOPs")
#         print(f"────────────────────────────────────────────────────────────────\n")

#         return gf_total, gf_search, gf_track

#     except ImportError:
#         print("[GFLOPs] fvcore not found — run:  pip install fvcore")
#         return 0.0, 0.0, 0.0
#     except Exception as e:
#         print(f"[GFLOPs] profiling failed: {e}")
#         return 0.0, 0.0, 0.0
    
# def _confidence_color(score: float):
#     s = max(0.0, min(1.0, score))
#     return (0, int(255 * s), int(255 * (1.0 - s)))


# def _stamp_panel(panel: np.ndarray, label: str, updated: bool) -> None:
#     bar_color = (0, 70, 0) if updated else (30, 30, 30)
#     cv2.rectangle(panel, (0, 0), (panel.shape[1], 22), bar_color, -1)
#     cv2.putText(panel, label, (5, 15), FONT, 0.38, (255, 255, 255), 1, cv2.LINE_AA)


# def _resize_into(src: np.ndarray, dst: np.ndarray) -> None:
#     np.copyto(dst, cv2.resize(src, (dst.shape[1], dst.shape[0])))


# def _draw_legend(canvas: np.ndarray, x: int, y: int) -> None:
#     entries = [
#         (C_GT,           "GT bbox (init)"),
#         ((0, 255, 0),    "Pred  conf=1.0"),
#         ((0, 128, 255),  "Pred  conf=0.5"),
#         ((0, 0, 255),    "Pred  conf=0.0"),
#         (C_SEARCH,       "Search region"),
#         ((0, 165, 255),  "YOLO ROI"),
#     ]
#     for i, (color, text) in enumerate(entries):
#         iy = y + i * 16
#         cv2.rectangle(canvas, (x, iy - 9), (x + 12, iy + 3), color, -1)
#         cv2.putText(canvas, text, (x + 16, iy), FONT, 0.35,
#                     (210, 210, 210), 1, cv2.LINE_AA)


# def _refresh_panels(tracker, dyn_image, panel_template, panel_search,
#                     frame_idx, updated):
#     from utils.utils import extend_bbox, get_extended_crop

#     dyn_bbox = tracker.running_dynamic_bbox
#     cfg      = tracker.tracking_config
#     ih, iw   = dyn_image.shape[:2]
#     pad_val  = np.mean(dyn_image, axis=(0, 1))

#     t_ctx = extend_bbox(dyn_bbox, image_width=iw, image_height=ih,
#                         offset=cfg["template_bbox_offset"])
#     t_crop, _, _ = get_extended_crop(image=dyn_image, bbox=dyn_bbox, context=t_ctx,
#                                      crop_size=cfg["template_size"],
#                                      padding_value=pad_val)
#     _resize_into(t_crop, panel_template)
#     _stamp_panel(panel_template, f"TEMPLATE  F:{frame_idx}", updated=updated)

#     s_ctx = extend_bbox(dyn_bbox, image_width=iw, image_height=ih,
#                         offset=cfg["search_context"])
#     s_crop, _, _ = get_extended_crop(image=dyn_image, bbox=dyn_bbox, context=s_ctx,
#                                      crop_size=cfg["instance_size"],
#                                      padding_value=pad_val)
#     _resize_into(s_crop, panel_search)
#     _stamp_panel(panel_search, "SEARCH CTX", updated=updated)


# # ──────────────────────────────────────────────────────────────────────────────
# # Main inference loop
# # ──────────────────────────────────────────────────────────────────────────────

# def run_inference(
#     initial_bbox,
#     video_path: str,
#     tracker,
#     output_path: str = "outputs/tracked_video.mp4",
#     output_video: bool = True,
#     plot_debug: bool = False,
#     plot_every_n: int = 1,
#     save_debug_frames: bool = True,
#     measure_gflops: bool = True,
# ):
#     """
#     Identical to the original run_inference, with one new parameter:

#     measure_gflops : bool  (default True)
#         Profile one full tracker.update() call on frame 1 via torch.profiler
#         and report GFLOPs at the end.  Set False to skip entirely.
#     """
#     is_dam        = hasattr(tracker, 'tracker')
#     inner_tracker = tracker.tracker if is_dam else tracker

#     initial_bbox = np.array(initial_bbox).astype(int)

#     os.makedirs(os.path.dirname(output_path), exist_ok=True)
#     head, tail = os.path.split(output_path)
#     bbox_dir   = os.path.join(head, "bboxes")
#     os.makedirs(bbox_dir, exist_ok=True)
#     bbox_file  = os.path.join(bbox_dir, os.path.splitext(tail)[0] + ".txt")

#     cap = cv2.VideoCapture(video_path)
#     if not cap.isOpened():
#         raise RuntimeError(f"Cannot open video: {video_path}")

#     fps = cap.get(cv2.CAP_PROP_FPS)
#     int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

#     ret, first_bgr = cap.read()
#     if not ret or first_bgr is None:
#         cap.release()
#         raise RuntimeError(f"Cannot read first frame: {video_path}")

#     h, w = first_bgr.shape[:2]

#     # ── Debug-plot helpers ───────────────────────────────────────────────────

#     if plot_debug:
#         import matplotlib.pyplot as plt
#         try:
#             from IPython.display import clear_output, display
#             _ipython_available = True
#         except ImportError:
#             _ipython_available = False

#         def _to_rgb(img):
#             if img is None:
#                 return np.full((128, 128, 3), 80, dtype=np.uint8)
#             if img.ndim == 2:
#                 return np.stack([img] * 3, axis=-1)
#             if img.shape[2] == 3:
#                 return img[:, :, ::-1].copy()
#             return img[:, :, :3][:, :, ::-1]

#         def _crop_roi(frame, bbox_xywh):
#             if bbox_xywh is None:
#                 return None
#             x, y, bw, bh = map(int, bbox_xywh)
#             x1 = max(0, x);  y1 = max(0, y)
#             x2 = min(frame.shape[1], x + bw)
#             y2 = min(frame.shape[0], y + bh)
#             return frame[y1:y2, x1:x2] if x2 > x1 and y2 > y1 else None

#         def _plot_debug_frame(frame_idx, frame, bbox, score, in_occlusion,
#                               yolo_dets, mapping, debug_frames_dir="debug_frames"):
#             from utils.utils import extend_bbox, get_extended_crop
#             cfg = inner_tracker.tracking_config

#             def _tmpl_crop(src_img, src_bbox):
#                 if src_img is None or src_bbox is None:
#                     return None
#                 try:
#                     ih, iw  = src_img.shape[:2]
#                     pad_val = np.mean(src_img, axis=(0, 1))
#                     t_ctx   = extend_bbox(src_bbox, image_width=iw, image_height=ih,
#                                           offset=cfg["template_bbox_offset"])
#                     crop, _, _ = get_extended_crop(image=src_img, bbox=src_bbox,
#                                                    context=t_ctx,
#                                                    crop_size=cfg["template_size"],
#                                                    padding_value=pad_val)
#                     return crop
#                 except Exception:
#                     return None

#             static_tmpl = _tmpl_crop(
#                 tracker.init_frame if is_dam else None,
#                 tracker.init_bbox  if is_dam else getattr(inner_tracker, '_init_rect', None),
#             )
#             dyn_tmpl = _tmpl_crop(
#                 inner_tracker.running_dynamic_image,
#                 inner_tracker.running_dynamic_bbox,
#             )
#             search_crop = _crop_roi(frame, mapping)

#             dyn_search = None
#             try:
#                 dyn_img  = inner_tracker.running_dynamic_image
#                 dyn_bbox = inner_tracker.running_dynamic_bbox
#                 if dyn_img is not None and dyn_bbox is not None:
#                     ih, iw  = dyn_img.shape[:2]
#                     pad_val = np.mean(dyn_img, axis=(0, 1))
#                     s_ctx   = extend_bbox(dyn_bbox, image_width=iw, image_height=ih,
#                                           offset=cfg["search_context"])
#                     dyn_search, _, _ = get_extended_crop(
#                         image=dyn_img, bbox=dyn_bbox, context=s_ctx,
#                         crop_size=cfg["instance_size"], padding_value=pad_val)
#             except Exception:
#                 pass

#             annot = frame.copy()
#             if mapping is not None:
#                 mx, my, mw, mh = map(int, mapping)
#                 cv2.rectangle(annot, (mx, my), (mx + mw, my + mh), (0, 165, 255), 1)
#             if in_occlusion and yolo_dets:
#                 held = getattr(tracker, 'held_box', None) if is_dam else None
#                 for det in yolo_dets:
#                     dx, dy, dw, dh = map(int, det)
#                     is_dist = held is not None and _iou(det, held) >= getattr(tracker, 'tau_occ', 0.3)
#                     col = (0, 0, 180) if is_dist else (0, 200, 255)
#                     cv2.rectangle(annot, (dx, dy), (dx + dw, dy + dh), col, 1)
#                     cv2.putText(annot, 'D' if is_dist else 'Y',
#                                 (dx + 2, dy + 12), FONT, 0.38, col, 1, cv2.LINE_AA)
#             bx, by, bw, bh = map(int, bbox)
#             pred_col = (0, 0, 220) if in_occlusion else _confidence_color(score)
#             cv2.rectangle(annot, (bx, by), (bx + bw, by + bh), pred_col, 2)
#             cv2.putText(annot, f'{score:.2f}', (bx, max(by - 4, 12)),
#                         FONT, 0.5, pred_col, 1, cv2.LINE_AA)
#             if is_dam and in_occlusion:
#                 rx, ry, rw, rh = tracker._get_yolo_search_roi(frame)
#                 cv2.rectangle(annot, (rx, ry), (rx + rw, ry + rh), (0, 165, 255), 1)
#                 cv2.putText(annot, 'YOLO ROI', (rx + 2, ry + 12),
#                             FONT, 0.38, (0, 165, 255), 1, cv2.LINE_AA)

#             if save_debug_frames:
#                 os.makedirs(debug_frames_dir, exist_ok=True)
#                 fid = f"{frame_idx:06d}"
#                 for name, img in [
#                     ("template",         static_tmpl),
#                     ("dynamic_template", dyn_tmpl),
#                     ("search_roi",       search_crop),
#                     ("dynamic_search",   dyn_search),
#                     ("frame_annotated",  annot),
#                 ]:
#                     out = img if img is not None else np.full((128, 128, 3), 80, dtype=np.uint8)
#                     if out.ndim == 2:
#                         out = np.stack([out] * 3, axis=-1)
#                     cv2.imwrite(os.path.join(debug_frames_dir, f"{name}_{fid}.png"), out)

#             fig, axes = plt.subplots(1, 5, figsize=(20, 4),
#                                      gridspec_kw={'wspace': 0.12})
#             fig.patch.set_facecolor('#1a1a1a')
#             titles = ['Template (static)', 'Dynamic template', 'Search ROI',
#                       'Dynamic search',
#                       f'Frame {frame_idx}  |  score={score:.3f}'
#                       + ('  [OCCLUDED]' if in_occlusion else '')]
#             images = [_to_rgb(static_tmpl), _to_rgb(dyn_tmpl), _to_rgb(search_crop),
#                       _to_rgb(dyn_search), _to_rgb(annot)]
#             border_colors = ['#4a90d9', '#e67e22', '#27ae60', '#8e44ad',
#                              '#c0392b' if in_occlusion else '#2ecc71']
#             for ax, img, title, bcol in zip(axes, images, titles, border_colors):
#                 ax.imshow(img, interpolation='nearest')
#                 ax.set_title(title, fontsize=9, color='#e0e0e0', pad=4)
#                 ax.axis('off')
#                 for spine in ax.spines.values():
#                     spine.set_edgecolor(bcol); spine.set_linewidth(2); spine.set_visible(True)
#                 ax.set_facecolor('#1a1a1a')
#             if save_debug_frames:
#                 fig.savefig(os.path.join(debug_frames_dir,
#                                          f"composite_{frame_idx:06d}.png"),
#                             dpi=100, bbox_inches='tight',
#                             facecolor=fig.get_facecolor())
#             if _ipython_available:
#                 clear_output(wait=True); display(fig); plt.close(fig)
#             else:
#                 plt.tight_layout(); plt.show(block=False)
#                 plt.pause(0.001); plt.close(fig)

#     # ── Video writer setup ───────────────────────────────────────────────────

#     if output_video:
#         C_OCCLUDED        = (0, 0, 220)
#         C_YOLO_CANDIDATE  = (0, 200, 255)
#         C_YOLO_DISTRACTOR = (0, 0, 180)
#         C_STATUS_OCC      = (0, 0, 200)
#         C_STATUS_OK       = (0, 180, 0)
#         C_STATUS_TEXT     = (255, 255, 255)

#         canvas_h = max(h, PANEL_W * 2 + GAP)
#         total_w  = w + PANEL_W
#         y_off    = (canvas_h - h) // 2

#         canvas         = np.zeros((canvas_h, total_w, 3), dtype=np.uint8)
#         panel_template = np.zeros((PANEL_W, PANEL_W, 3), dtype=np.uint8)
#         panel_search   = np.zeros((PANEL_W, PANEL_W, 3), dtype=np.uint8)
#         row_tmpl   = np.s_[0:PANEL_W, w:total_w]
#         row_search = np.s_[PANEL_W + GAP:PANEL_W * 2 + GAP, w:total_w]

#         avi_path = os.path.splitext(output_path)[0] + "_tmp.avi"
#         writer   = cv2.VideoWriter(avi_path, cv2.VideoWriter_fourcc(*"XVID"),
#                                    fps, (total_w, canvas_h))

#         def _draw_status_pill(canvas, in_occlusion):
#             label = "OCCLUDED" if in_occlusion else "TRACKING"
#             color = C_STATUS_OCC if in_occlusion else C_STATUS_OK
#             px, py, pw, ph = 8, y_off + 8, 140, 28
#             r = ph // 2
#             cv2.rectangle(canvas, (px + r, py), (px + pw - r, py + ph), color, -1)
#             cv2.circle(canvas, (px + r, py + r), r, color, -1)
#             cv2.circle(canvas, (px + pw - r, py + r), r, color, -1)
#             (tw, th), _ = cv2.getTextSize(label, FONT, 0.50, 1)
#             cv2.putText(canvas, label,
#                         (px + (pw - tw) // 2, py + (ph + th) // 2 - 1),
#                         FONT, 0.50, C_STATUS_TEXT, 1, cv2.LINE_AA)

#     # ── Initialise tracker ───────────────────────────────────────────────────
#     # system_gflops   = 0.0
#     prefetched_frame = None
#     tracker.initialize(first_bgr, initial_bbox)
#     if measure_gflops:
#          system_gflops, gflops_search, gflops_track = _measure_gflops_inference(
#         inner_tracker.net,
#         tracker.tracking_config,
#         device="cuda" if next(inner_tracker.net.parameters()).is_cuda else "cpu",
#     )

#     if output_video:
#         _refresh_panels(inner_tracker, first_bgr, panel_template, panel_search,
#                         frame_idx=0, updated=True)
#         canvas.fill(0)
#         canvas[y_off:y_off + h, :w] = first_bgr
#         canvas[row_tmpl]   = panel_template
#         canvas[row_search] = panel_search
#         bx, by, bw, bh = map(int, initial_bbox)
#         cv2.rectangle(canvas, (bx, by + y_off), (bx + bw, by + bh + y_off), C_GT, 2)
#         _draw_status_pill(canvas, in_occlusion=False)
#         cv2.putText(canvas, "F:0  INIT", (156, y_off + 22),
#                     FONT, 0.45, C_FRAME, 1, cv2.LINE_AA)
#         _draw_legend(canvas, w + 4, canvas_h - 90)
#         writer.write(canvas)
#         last_dyn_bbox = inner_tracker.running_dynamic_bbox.copy()

#     if plot_debug:
#         _plot_debug_frame(frame_idx=0, frame=first_bgr, bbox=initial_bbox,
#                           score=1.0, in_occlusion=False, yolo_dets=[], mapping=None)

#     # ── GFLOPs: profile one full update() call on frame 1 ───────────────────
#     # Read frame 1 early, run the profiler on it, then re-use it as the first
#     # iteration of the main loop so nothing is skipped.



#     # if measure_gflops:
#     #     ret_pf, prefetched_frame = cap.read()
#     #     if ret_pf and prefetched_frame is not None:
#     #         print("[GFLOPs] Profiling full system on frame 1 …")
#     #         system_gflops = _measure_gflops(tracker.update, prefetched_frame)
#     #         if system_gflops > 0:
#     #             print(f"[GFLOPs] {system_gflops:.4f} GFLOPs / frame")
#     #         else:
#     #             print("[GFLOPs] Measurement returned 0 — op types may be unsupported.")

#     # ── Main loop ────────────────────────────────────────────────────────────

#     tracked_bboxes  = [initial_bbox]
#     times_normal    = []
#     times_occlusion = []
#     frame_times     = []

#     import time

#     try:
#         frame_idx = 1
#         while True:
#             ret, frame = cap.read()
#             if not ret or frame is None:
#                 break

#             t0 = time.perf_counter()
#             result     = tracker.update(frame)
#             elapsed_ms = (time.perf_counter() - t0) * 1000.0
#             frame_times.append(elapsed_ms)

#             bbox         = result[0]
#             score        = result[1]
#             in_occlusion = result[2] if is_dam else False
#             yolo_dets    = result[3] if (is_dam and len(result) > 3) else []

#             tracked_bboxes.append(bbox)
#             (times_occlusion if in_occlusion else times_normal).append(elapsed_ms)

#             if plot_debug and (frame_idx % plot_every_n == 0):
#                 mapping = getattr(inner_tracker.tracking_state, 'mapping', None)
#                 _plot_debug_frame(frame_idx=frame_idx, frame=frame, bbox=bbox,
#                                   score=score, in_occlusion=in_occlusion,
#                                   yolo_dets=yolo_dets, mapping=mapping)

#             if output_video:
#                 cur_dyn_bbox     = inner_tracker.running_dynamic_bbox
#                 cur_dyn_obj      = inner_tracker.running_dynamic_image
#                 template_updated = not np.array_equal(cur_dyn_bbox, last_dyn_bbox)

#                 if template_updated:
#                     _refresh_panels(inner_tracker, cur_dyn_obj, panel_template,
#                                     panel_search, frame_idx=frame_idx, updated=True)
#                     last_dyn_bbox = cur_dyn_bbox.copy()
#                 else:
#                     _stamp_panel(panel_template,
#                                  f"Dynamic TEMPLATE  F:{frame_idx - 1}", updated=False)
#                     _stamp_panel(panel_search, "SEARCH CTX", updated=False)

#                 canvas.fill(0)
#                 canvas[y_off:y_off + h, :w] = frame
#                 canvas[row_tmpl]   = panel_template
#                 canvas[row_search] = panel_search

#                 if in_occlusion:
#                     cv2.rectangle(canvas, (w, 0), (total_w - 1, canvas_h - 1),
#                                   C_OCCLUDED, 3)
#                 elif template_updated:
#                     cv2.rectangle(canvas, (w, 0), (total_w - 1, canvas_h - 1),
#                                   C_UPDATE, 2)

#                 mapping = inner_tracker.tracking_state.mapping
#                 if mapping is not None:
#                     mx, my, mw, mh = map(int, mapping)
#                     cv2.rectangle(canvas, (mx, my + y_off), (mx + mw, my + mh + y_off),
#                                   C_SEARCH, 1)

#                 if in_occlusion and yolo_dets:
#                     held = tracker.held_box if is_dam else None
#                     for det in yolo_dets:
#                         dx, dy, dw, dh = map(int, det)
#                         is_dist = held is not None and _iou(det, held) >= tracker.tau_occ
#                         color   = C_YOLO_DISTRACTOR if is_dist else C_YOLO_CANDIDATE
#                         cv2.rectangle(canvas, (dx, dy + y_off), (dx + dw, dy + dh + y_off),
#                                       color, 1)
#                         cv2.putText(canvas, "D" if is_dist else "Y",
#                                     (dx + 2, dy + y_off + 12),
#                                     FONT, 0.38, color, 1, cv2.LINE_AA)

#                 bx, by, bw, bh = map(int, bbox)
#                 pred_color = C_OCCLUDED if in_occlusion else _confidence_color(score)
#                 cv2.rectangle(canvas, (bx, by + y_off), (bx + bw, by + bh + y_off),
#                               pred_color, 2)
#                 cv2.putText(canvas, f"{score:.2f}", (bx, max(by + y_off - 4, 12)),
#                             FONT, 0.45, pred_color, 1, cv2.LINE_AA)

#                 _draw_status_pill(canvas, in_occlusion=in_occlusion)

#                 hud = (f"F:{frame_idx}  [DAM RECOVERY]" if in_occlusion
#                        else f"F:{frame_idx}  [TMPL UPDATE]" if template_updated
#                        else f"F:{frame_idx}")
#                 cv2.putText(canvas, hud, (156, y_off + 22),
#                             FONT, 0.45, C_FRAME, 1, cv2.LINE_AA)

#                 _draw_legend(canvas, w + 4, canvas_h - 90)

#                 if is_dam and in_occlusion:
#                     rx, ry, rw, rh = tracker._get_yolo_search_roi(frame)
#                     cv2.rectangle(canvas, (rx, ry + y_off), (rx + rw, ry + rh + y_off),
#                                   (0, 165, 255), 1)
#                     cv2.putText(canvas, "YOLO ROI", (rx + 2, ry + y_off + 12),
#                                 FONT, 0.38, (0, 165, 255), 1, cv2.LINE_AA)

#                 writer.write(canvas)

#             frame_idx += 1

#     finally:
#         cap.release()
#         if output_video:
#             writer.release()

#     if output_video and avi_path != output_path:
#         os.replace(avi_path, output_path)

#     with open(bbox_file, "w", encoding="utf-8") as f:
#         for bb in tracked_bboxes:
#             f.write(f"{bb[0]} {bb[1]} {bb[2]} {bb[3]}\n")

#     if output_video:
#         del canvas, panel_template, panel_search

#     # ── Latency + GFLOPs report ──────────────────────────────────────────────

#     def _stats(name, arr):
#         if not arr:
#             print(f"{name:20s}  no data"); return
#         a = np.array(arr)
#         print(f"{name:20s}  n={len(a):4d}  "
#               f"mean={a.mean():.1f}ms  med={np.median(a):.1f}ms  "
#               f"p95={np.percentile(a, 95):.1f}ms  p99={np.percentile(a, 99):.1f}ms  "
#               f"min={a.min():.1f}ms  max={a.max():.1f}ms  fps={1000/a.mean():.1f}")

#     print("\n─── Latency Report ───────────────────────────────────────")
#     _stats("ALL FRAMES",   frame_times)
#     _stats("NORMAL TRACK", times_normal)
#     _stats("OCCLUSION",    times_occlusion)
#     if times_occlusion:
#         print(f"  occlusion frames: {len(times_occlusion)} "
#               f"({100 * len(times_occlusion) / len(frame_times):.1f}% of total)")

#     if measure_gflops:
#         print("\n─── GFLOPs Report (inference-only ops) ───────────────────")
#         if system_gflops > 0:
#             n      = len(frame_times)
#             mean_s = np.mean(frame_times) / 1000.0 if frame_times else None
#             print(f"  Search encoder+neck (every frame) : {gflops_search:.4f} G")
#             print(f"  Attention + head    (every frame) : {gflops_track:.4f} G")
#             print(f"  Per-frame total                   : {system_gflops:.4f} G")
#             print(f"  Total frames                      : {n}")
#             print(f"  Total GFLOPs                      : {system_gflops * n:.2f} G")
#             if mean_s:
#                 print(f"  GFLOP/s @ mean latency           : {system_gflops / mean_s:.2f}")
#         else:
#             print("  GFLOPs measurement unavailable.")

#     print("──────────────────────────────────────────────────────────\n")

#     gc.collect()

In [3]:
video_paths = [
    "/home/moha/AIC-4/data_competition/dataset2/Gull2/Gull2_24.mp4",
    "/home/moha/AIC-4/data_competition/dataset3/car16_3/car16_3_24.mp4",
    "/home/moha/AIC-4/data_competition/dataset2/RaceCar1/RaceCar1_24.mp4",
    "/home/moha/AIC-4/data_competition/dataset5/uav2/uav2_30.mp4",
    "/home/moha/AIC-4/data_competition/dataset5/uav7/uav7_30.mp4",
    "/home/moha/AIC-4/data_competition/dataset2/RcCar6/RcCar6_24.mp4",
    "/home/moha/AIC-4/data_competition/dataset2/Motor1/Motor1_24.mp4",
    "/home/moha/AIC-4/data_competition/dataset2/MountainBike5/MountainBike5_30.mp4",
    "/home/moha/AIC-4/data_competition/dataset1/person_3/person_3.mp4",
    "/home/moha/AIC-4/data_competition/dataset3/truck/truck_30.mp4",
    "/home/moha/AIC-4/data_competition/dataset3/human3/human3_24.mp4",
    "/home/moha/AIC-4/data_competition/dataset1/sheeps_2/sheeps_2.mp4",
    "/home/moha/AIC-4/data_competition/dataset2/Animal3/Animal3_30.mp4",
    "/home/moha/AIC-4/data_competition/dataset5/person16/person16_96.mp4",
    "/home/moha/AIC-4/data_competition/dataset3/electric_box/electric_box_24.mp4",
    "/home/moha/AIC-4/data_competition/dataset2/RcCar4/RcCar4_24.mp4",
    "/home/moha/AIC-4/data_competition/dataset4/group2/group2_96.mp4",
    "/home/moha/AIC-4/data_competition/dataset3/jogging2/jogging2_30.mp4",
    "/home/moha/AIC-4/data_competition/dataset3/car6_2/car6_2_30.mp4",
    "/home/moha/AIC-4/data_competition/dataset4/person19/person19_96.mp4",
    "/home/moha/AIC-4/data_competition/dataset3/couple/couple_24.mp4",
    "/home/moha/AIC-4/data_competition/dataset3/tennis_player1_2/tennis_player1_2_30.mp4",
    "/home/moha/AIC-4/data_competition/dataset5/car4/car4_96.mp4"
]

In [4]:
import os
import numpy as np
from omegaconf import OmegaConf
from models.SiamABC.tracker.trt_engine.siamabc import get_trt_tracker

weights_path = "/home/moha/AIC-for submission/SiamRAM/checkpoints/head_epoch_000.pth"
yaml_config_path = "/home/moha/AIC-for submission/SiamRAM/config/inference_config.yaml" 

config = OmegaConf.load(yaml_config_path)

# wrapped = get_trt_tracker(
#     config=config,
#     weights_path=weights_path,
#     **config.trt_engine  
# )


# tracker = SiamRAMTracker(
#     siam_tracker=wrapped,
#     **config.ram_tracker 
# )


# start = 1
# for i in range(start, start + len(video_paths)):
#     video_path = video_paths[i - start]
#     ann_path = os.path.join(os.path.dirname(video_path), "annotation.txt")
#     output_path = f"outputs/test_5_after/test_{i}.mp4"

#     # Load initial bbox
#     init_bbox = np.loadtxt(ann_path, delimiter=",", dtype=np.float32).tolist()
#     if not isinstance(init_bbox[0], (int, float)):
#         init_bbox = init_bbox[0]
        
#     print(f"Processing Video {i}: {video_path}")

#     run_inference(
#         video_path=video_path,
#         initial_bbox=init_bbox,
#         tracker=tracker,
#         output_path=output_path
#     )

In [5]:
config

{'model': {'_target_': 'models.SiamABC.model.SiamABC.SiamABCNet', 'backbone': 'custom_fbnet', 'growth_factor': 1.2, 'upsample': 'pixel_shuffle', 'pretrained': True, 'num_classes': 1, 'num_filters': 32, 'num_channels': 3, 'align': False, 'img_size': 256, 'stride': 2, 'conv_block': 'sep_conv', 'towernum': 2, 'model_size': 'M', 'max_layer': 4, 'crop_template_features': False}, 'tracker': {'_target_': 'models.SiamABC.tracker.SiamABC_Tracker.SiamABCTracker', 'penalty_k': 0.062, 'window_influence': 0.38, 'lr': 0.4, 'windowing': 'cosine', 'total_stride': 16, 'score_size': 16, 'N': 10, 'dynamic_update': True, 'similarity_score': False, 'stride': 2, 'smooth': False, 'bbox_ratio': 0.5, 'template_bbox_offset': 0.2, 'search_context': 2, 'instance_size': 256, 'template_size': 128, 'memory_window_size': 20, 'dynamic_update_threshold': 0.87, 'running_confidence_floor_value': 100, 'iou_threshold': 0.6, 'warmup_frames': 35, 'warmup_window_size': 8}, 'trt_engine': {'lambda_tta': 0.1, 'fp16': True, 'cuda

In [6]:
# dunamic_search = 4.3760
# search = 4.3760   
# dynamic_template = 1.0940
# template = 1.0940
# attention = 0.3865

# s = 0
# range_ = 200
# for i in range(range_):
#     if i==0:
#         s+=template + dynamic_template + dunamic_search + search + attention*2
#     elif i%10==0:
#         s+=dynamic_template + dunamic_search + search + attention*2
#     else:
#         s+=search + attention

# s/range_


In [7]:
9.5115/3


3.1705

In [10]:
config.trt_engine

{'lambda_tta': 0.1, 'fp16': True, 'cuda_id': 0}

In [12]:
# 0.768 score on public lb
test_public_lb = manifest["public_lb"]
with open("/home/moha/AIC-4/data_competition/metadata/contestant_manifest.json", "r") as f:
    manifest = json.load(f)
manifest.keys()
test_public_lb = manifest["public_lb"]

# test_pub_2 = {k:v for k , v in manifest["public_lb"].items() if v["dataset"]=="dataset2"}


folder_names = [
    # "uav2",
    # "uav7",
    # "RcCar6",
    "Motor1",
    "MountainBike5",
    "person_3",
    "truck",
    "human3",
    "sheeps_2",
    "Animal3",
    "person16",
    "electric_box",
    "RcCar4",
    "group2",
    "truck",
    "jogging2",
    "car6_2",
    "person19",
    "couple",
    "tennis_player1_2",
    "car4",
]

data_dir = "../../AIC-4/data_competition"
outputs_dir = "outputs/SiamDAM__test"
test_public_lb = {
k: v for k, v in manifest["public_lb"].items()
if v["dataset"] in [
    "dataset1",
    "dataset2",
    "dataset3",
    "dataset4",
    "dataset5"
]
# and v["seq_name"] in  folder_names
}
model_size = "M"
weights_path = "/home/moha/AIC-for submission/SiamRAM/checkpoints/head_epoch_000.pth"
# weights_path = "/home/moha/AIC-4/checkpoints/bbox_head/all_datasets_finetuning/part_2/head_epoch_001.pth"
config_path = "../../AIC-4/external/SiamABC/core/config"
config_name = "SiamABC_tracker"

# config = load_hydra_config_from_path(config_path=config_path, config_name=config_name)
# config["model"]["model_size"] = 'S' if model_size=="S_Tiny" else 'M'
# config["tracker"]["N"] =10
# config["tracker"]["lr"] = 0.4
# config["tracker"]["dynamic_update"] = True 
# config["tracker"]["memory_window_size"] = 20
# config["tracker"]["dynamic_update_threshold"] = 0.87
# config["tracker"]["running_confidence_floor_value"] = 100
# config["tracker"]["search_context"] = 2
# config["tracker"]["iou_threshold"] = 0.6
# config["model"]["_target_"] = "models.SiamABC.model.SiamABC.SiamABCNet"
# config["tracker"]["_target_"] = "models.SiamABC.tracker.SiamABC_Tracker.SiamABCTracker"
# config["tracker"]["warmup_frames"] = 35
# config["tracker"]["warmup_window_size"] = 8

# config["tracker"]["jump_center_shift"] = 0.7
# config["tracker"]["jump_size_ratio"] = 4
# config["tracker"]["jump_score_override"] = 0.99



# config["tracker"]["window_influence"] =  0.45 
# config["tracker"]["penalty_k"] = 0.10


# wrapped = get_tracker(config=config, weights_path=weights_path , lambda_tta=0.1 , continuous=False)



wrapped = get_trt_tracker(
    config=config,
    weights_path=weights_path,
    **config.trt_engine  
)


# siam.all_memory_imgs = deque(maxlen=250)   # was 5000
# siam.classification_scores = deque(maxlen=250)


tracker = SiamRAMTracker(
        siam_tracker        = wrapped,
        yolo_weights        = "yolo11n.pt",

        # ── Core thresholds ───────────────────────────────────────────────────
        conf_threshold      = 0.55,      # score BELOW this starts the entry streak
        reacq_threshold     = 0.7,      # tracker score to exit occlusion (phase 0)
        occ_siam_reacq_threshold = 0.80,
        yolo_conf           = 0.3,
        app_match_threshold = 0.1,      # cosine sim to accept tracker bbox as target
        occ_siam_margin=0.5,
        nudge_alpha         = 0.0,

        # ── NEW: Occlusion entry hysteresis ───────────────────────────────────
        entry_patience      = 1,        # N consecutive bad frames before occlusion

        # ── NEW: Multi-frame candidate collection ─────────────────────────────
        cand_collection_frames = 1,     # YOLO collection frames before final DRM

        # ── NEW: Velocity scoring weight & guard ──────────────────────────────
        drm_lam_cand_vel    = 0.0,     # weight of vel_score in final DRM phase
                                        # set 0 to disable velocity scoring
        vel_score_min_speed = 0.5,      # min EKF speed (px/frame) to use vel score
                                        # if EKF speed < this → vel_score = 0.5 (neutral)

        # ── NEW: Tiny / long-distance object ROI parameters ───────────────────
        # Used automatically when _is_long_distance() returns True.
        long_distance_area_fraction =  0.030,
        tiny_roi_start_expand            = 8.0,
        tiny_yolo_search_expand          = 20.0,
        tiny_search_expand_growth_factor = 1.3,
        tiny_search_expand_growth_every  = 50,
        tiny_search_expand_max           = 40.0,

        # ── DRM (existing params) ─────────────────────────────────────────────
        drm_tau_sim         = 0.6,
        mem_capacity        = 50,
        drm_mmin            = 3,
        drm_capacity        = 20,
        history_decay       = 0.1,
        drm_lam_dist        = 0.0,
        drm_lam_cand_dir    = 0.0,
        drm_lam_time        = 0.0,
        drm_lam_app         = 1,
        drm_lam_iou         = 0.0,
        drm_lam_mot         = 0.0,
        drm_margin          = 0.6,
        drm_skip_threshold  = 23,
        drm_top_k           = 100,
        vel_dir_hard_gate = 0.4,   # |cos| threshold below which score → 0.05
        yolo_filter_class  = False, # filter candidates to target class
        yolo_class_detect_frames   = 5,     # stride 

        # ── History ───────────────────────────────────────────────────────────
        conf_history_len    = 200,
        size_history_len    = 200,
        history_skip_last   = 8,

        # ── ROI expansion (normal objects) ────────────────────────────────────
        roi_start_expand                 = 20,
        yolo_search_expand               = 100,
        search_expand_growth_factor      = 1.4,
        search_expand_growth_every       = 150,
        search_expand_max                = 500.0,

        # ── Misc ──────────────────────────────────────────────────────────────
        velocity_window_average          = 200,
        shrinkage_max_lookback           = 30,
        enter_occlusion_on_loss          = True,
        drm_gamma                        = 0,
        debug=False

    )
for i, (key, value) in enumerate(test_public_lb.items()):

    print(f"Processing video {i + 1}/{len(test_public_lb)}: {value['video_path']}")
    video_path = os.path.join(
        data_dir, value["video_path"]
    )
    ann_path = os.path.join(
        data_dir, value["annotation_path"]
    )

    output_path = os.path.join(
        outputs_dir, value["video_path"]
    )

    init_bbox = np.loadtxt(ann_path, delimiter=",", dtype=np.float32).tolist()

    run_inference(
        video_path=video_path,
        initial_bbox=init_bbox,
        tracker=tracker,
        output_path=output_path
    )

with open("/home/moha/AIC-4/data_competition/metadata/contestant_manifest.json", "r") as f:
    test_public_lb = json.load(f)["public_lb"]
submission_df = defaultdict(list)
for key, value in test_public_lb.items():
    video_path = os.path.join(data_dir, value["video_path"])
    output_path = os.path.join(outputs_dir, value["video_path"])
    print(video_path, output_path)

    head, tail = os.path.split(output_path)
    bbox_dir = os.path.join(head, 'bboxes')
    bbox_file = os.path.join(bbox_dir, os.path.splitext(tail)[0] + '.txt')

    seq_id = os.path.splitext(value["video_path"])[0]
    if os.path.exists(bbox_file):
        with open(bbox_file, 'r') as f:
            lines = f.read().strip().split('\n')

            for frame_idx, line in enumerate(lines):
                x, y, w, h = line.strip().split()
                submission_df["id"].append(f"{key}_{frame_idx}")
                submission_df["x"].append(float(x))
                submission_df["y"].append(float(y))
                submission_df["w"].append(float(w))
                submission_df["h"].append(float(h))

submission = pd.DataFrame(submission_df)
submission.to_csv("submission.csv", index=False)

print(submission.head())
print(f"Total rows: {len(submission)}")

  warnings.warn(

  warnings.warn(msg)

INFO:torch_tensorrt [TensorRT Conversion Context]:[MemUsageChange] Init builder kernel library: CPU -269, GPU +0, now: CPU 5023, GPU 2115 (MiB)
INFO:torch_tensorrt [TensorRT Conversion Context]:Global timing cache in use. Profiling results in this builder pass will be stored.
INFO:torch_tensorrt [TensorRT Conversion Context]:Detected 1 inputs and 1 output network tensors.
INFO:torch_tensorrt [TensorRT Conversion Context]:Total Host Persistent Memory: 258240 bytes
INFO:torch_tensorrt [TensorRT Conversion Context]:Total Device Persistent Memory: 36864 bytes
INFO:torch_tensorrt [TensorRT Conversion Context]:Max Scratch Memory: 0 bytes
INFO:torch_tensorrt [TensorRT Conversion Context]:[BlockAssignment] Started assigning block shifts. This will take 46 steps to complete.
INFO:torch_tensorrt [TensorRT Conversion Context]:[BlockAssignment] Algorithm ShiftNTopDown took 0.652177ms to assign 3 blocks to 46 nodes requiring 1179648 bytes.
INFO:torch_tensorrt

{'penalty_k': 0.062, 'window_influence': 0.38, 'lr': 0.4, 'windowing': 'cosine', 'total_stride': 16, 'score_size': 16, 'N': 10, 'dynamic_update': True, 'similarity_score': False, 'stride': 2, 'smooth': False, 'bbox_ratio': 0.5, 'template_bbox_offset': 0.2, 'search_context': 2, 'instance_size': 256, 'template_size': 128, 'memory_window_size': 20, 'dynamic_update_threshold': 0.87, 'running_confidence_floor_value': 100, 'iou_threshold': 0.6, 'warmup_frames': 35, 'warmup_window_size': 8}
Processing video 1/89: dataset1/Car_video/Car_video.mp4

─── Latency Report ───────────────────────────────────────
ALL FRAMES            n= 584  mean=11.9ms  med=11.7ms  p95=16.0ms  p99=18.5ms  min=7.1ms  max=23.8ms  fps=84.0
NORMAL TRACK          n= 584  mean=11.9ms  med=11.7ms  p95=16.0ms  p99=18.5ms  min=7.1ms  max=23.8ms  fps=84.0
OCCLUSION             no data
──────────────────────────────────────────────────────────

Processing video 2/89: dataset1/Car_video_4/Car_video_4.mp4

─── Latency Report ───

KeyboardInterrupt: 

In [ ]:
submission.to_csv("submission.csv", index=False)

print(submission.head())
print(f"Total rows: {len(submission)}")

                     id      x      y      w      h
0  dataset1/Car_video_0  536.0  551.0  226.0  142.0
1  dataset1/Car_video_1  533.0  549.0  237.0  150.0
2  dataset1/Car_video_2  531.0  549.0  237.0  152.0
3  dataset1/Car_video_3  531.0  549.0  237.0  152.0
4  dataset1/Car_video_4  529.0  550.0  239.0  155.0
Total rows: 74293
